# Transformação de dados - Large Office

### Importando bibliotecas

In [45]:
import os
import pandas as pd


### Padronização de string

In [46]:
def std_codes(code):
    if str(code).replace('.','').isdigit():
        return(str(int(code))).replace('_x000D_\n', '').replace('\n', '')
    else:
        return str(code).replace('_x000D_\n', '').replace('\n', '')
    

#def std_patr(code):
#    if str(code).isdigit():
#        return str(int(code))
#    else:
#        return str(code)

### Importação de dados

In [47]:
# Atualizar caminho do dado de acordo com a máquina ultilizada!

# 1) Consolidação Dados de Consumo
pasta_rateio = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Raw\Consumo\Rateio"
pasta_telemetria = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Raw\Consumo\Telemetria"
arquivo_gramaturas = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Raw\Gramaturas\Controle Gramaturas Oficial v_40.xlsx"

# 2) Capacidade de Insumos por patrimônio
dados_path = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Raw\Rotas para Análise\Rotas SP - Kearney - Handover.xlsx"
capacidade = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Raw\Capacidade de Máquinas\Capacidade de Máquinas.xlsx"

#3) Consumo de Insumos por patrimônio
consumo_consolidado = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Raw\Consumo\Consumo_Consolidado.xlsx"
arquivo_gramaturas = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Raw\Gramaturas\Controle Gramaturas Oficial v_40.xlsx"


## 1) Consolidando Dados de Consumo (Rateio e Telemetria).

Para facilitar a consolidação é necessário deixar os arquivos em pastas diferentes sendo o nome de cada arquivo o Mês+Abreviação do Ano ("0725" para Julho de 2025, por exemplo)

In [48]:

# -----------------------------
# Função para converter nome do arquivo em DATA
# Exemplo: "0725" -> 2025-07-01
# -----------------------------
def parse_nome_arquivo(nome_base):
    mes = int(nome_base[:2])
    ano = int(nome_base[2:])
    ano = 2000 + ano if ano < 100 else ano  # converte 25 -> 2025
    return pd.Timestamp(year=ano, month=mes, day=1)

# -----------------------------
# Carregar SKUs válidos do Excel de Gramaturas
# -----------------------------
gramaturas_sku = pd.read_excel(arquivo_gramaturas, sheet_name='Gramaturas Kearney', skiprows=2, usecols="B:R")
skus = gramaturas_sku["SKU"].dropna().astype(str).unique().tolist()

# -----------------------------
# Processamento principal
# -----------------------------
arquivos_rateio = [f for f in os.listdir(pasta_rateio) if f.endswith(".xlsx")]

lista_dfs = []

for arquivo in arquivos_rateio:
    nome_base, ext = os.path.splitext(arquivo)
    caminho_rateio = os.path.join(pasta_rateio, arquivo)
    caminho_telemetria = os.path.join(pasta_telemetria, arquivo)
    
    # --- Ler arquivo de Rateio ---
    df_rateio = pd.read_excel(caminho_rateio)
    df_rateio = df_rateio[df_rateio["FATURADO"] == "SIM"]
    df_rateio = df_rateio[["CODPARC", "NOMEPARC", "CODBEM", "CODPROD", "PRODUTO", "CONSUMO"]].copy()
    df_rateio = df_rateio.rename(columns={"CONSUMO": "CONSUMO DOSES"})
    df_rateio["Data"] = parse_nome_arquivo(nome_base)
    
    # --- Ler arquivo de Telemetria correspondente ---
    if os.path.exists(caminho_telemetria):
        df_tel = pd.read_excel(caminho_telemetria)
        df_tel.columns = df_tel.columns.str.lower()
        
        # Filtrar linhas onde id_patrimonio não está em CODBEM do rateio
        df_tel = df_tel[~df_tel["id_patrimonio"].isin(df_rateio["CODBEM"])]
        
        # Selecionar colunas relevantes
        df_tel = df_tel[["id_patrimonio", "sku", "produto", "quantidade_total", "nomeparc"]]
        
        # Renomear colunas
        df_tel = df_tel.rename(columns={
            "id_patrimonio": "CODBEM",
            "sku": "CODPROD",
            "produto": "PRODUTO",
            "quantidade_total": "CONSUMO DOSES",
            "nomeparc": "NOMEPARC"
        })
        
        # 🔑 Filtrar apenas SKUs válidos
        df_tel = df_tel[df_tel["CODPROD"].astype(str).isin(skus)]
        
        # Garantir coluna CODPARC
        if "CODPARC" not in df_tel.columns:
            df_tel["CODPARC"] = None
        
        # Adicionar coluna DATA
        df_tel["Data"] = df_rateio["Data"].iloc[0] if not df_rateio.empty else pd.NaT
        
        # Unir os dois DataFrames
        df_unido = pd.concat([df_rateio, df_tel], ignore_index=True)
    else:
        df_unido = df_rateio
    
    lista_dfs.append(df_unido)

# -----------------------------
# Consolidação final
# -----------------------------
df_final = pd.concat(lista_dfs, ignore_index=True)

df_final["DATA"] = df_final["DATA"].dt.date

# Salvar resultado consolidado
saida = os.path.join(os.path.dirname(pasta_rateio), "Consumo_Consolidado.xlsx")
df_final.to_excel(saida, index=False)

print(f"✅ Processamento concluído! Arquivo salvo em: {saida}")


KeyboardInterrupt: 

## 2) Capacidade de Insumos por patrimônio

Aba de "insumo" em Dados.xlsx.

In [49]:
#Dados dos patrimonios
rotas_sp = pd.read_excel(dados_path, sheet_name="Rotas Atual", skiprows=1)

#Capacidade das máquinas
capacidade_df = pd.read_excel(capacidade, sheet_name='Capacidade', skiprows = 1)

In [50]:
#Filtrando os patrimônios não fixos
rotas_sp_filtrado = rotas_sp[rotas_sp["TIPO DE ATENDIMENTO CORRIGIDO"] != "FIXO"]


In [51]:
rotas_sp_filtrado

,FILIAL,CONTRATO,PARCEIRO,PATRIMÔNIO,"PATRIMONIO sem ""0""",MODELO,CAPACIDADE EM DOSES,TIPO DE MAQUINA,PDV,CLIENTE,...,HOSPITAL / LABORATÓRIO,Dia de Inventário,Seg,Ter,Qua,Qui,Sex,Sáb,Dom,Frequência Atual
0,SP,24347.0,60146.0,4862,4862,MAQ SUPER-AUTOMATICA SAECO INTELIA,100,BEBIDAS QUENTES,COPA TÉRREO,11° CARTÓRIO,...,NaN,QUINTA-FEIRA,X,X,X,X,X,NaN,NaN,5
1,SP,24347.0,60146.0,082001,82001,MAQ VENDING BQ NECTA BRIO 250,250,BEBIDAS QUENTES,TÉRREO,11° CARTÓRIO,...,NaN,QUINTA-FEIRA,X,X,X,X,X,NaN,NaN,5
2,SP,30504.0,908079.0,10653,10653,MAQ VENDING BQ NECTA COLIBRI C4,500,BEBIDAS QUENTES,COPA,CONSELHO TUTELAR - PERUS,...,NaN,TERÇA-FEIRA,X,NaN,X,NaN,X,NaN,NaN,3
3,SP,19571.0,73456.0,A16962,A16962,MAQ VENDING BQ COFFEEMAX III STD GER. II,120,BEBIDAS QUENTES,1 ANDAR,2ºCARTORIO,...,NaN,TERÇA-FEIRA,X,X,X,X,X,NaN,NaN,5
4,SP,35477.0,35614.0,028666,28666,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,10 ANDAR COPA,99 TAXIS,...,NaN,QUARTA-FEIRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3553,SP,38024.0,25158.0,031829,31829,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,NaN,YAMAHA JANDIRA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5
3554,SP,38024.0,25158.0,031831,31831,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,NaN,YAMAHA JANDIRA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5
3555,SP,38024.0,25158.0,034756,34756,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,NaN,YAMAHA JANDIRA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5
3556,SP,38024.0,25158.0,030905,30905,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,COPA,YAMAHA JANDIRA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5


In [52]:
#Filtrando os patrimônios não fixos
supervisor1= "ALAN FONTES"
supervisor2= "ANDRÉ"

rotas_sp_filtrado = rotas_sp_filtrado[rotas_sp_filtrado["SUPERVISOR"].isin([supervisor1, supervisor2])]


In [53]:
rotas_sp_filtrado

,FILIAL,CONTRATO,PARCEIRO,PATRIMÔNIO,"PATRIMONIO sem ""0""",MODELO,CAPACIDADE EM DOSES,TIPO DE MAQUINA,PDV,CLIENTE,...,HOSPITAL / LABORATÓRIO,Dia de Inventário,Seg,Ter,Qua,Qui,Sex,Sáb,Dom,Frequência Atual
4,SP,35477.0,35614.0,028666,28666,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,10 ANDAR COPA,99 TAXIS,...,NaN,QUARTA-FEIRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
5,SP,35477.0,35614.0,033465,33465,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,7 ANDAR,99 TAXIS,...,NaN,QUARTA-FEIRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
6,SP,35477.0,35614.0,28501,28501,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,7 ANDAR,99 TAXIS,...,NaN,QUARTA-FEIRA,X,X,X,X,X,NaN,NaN,5
7,SP,43179.0,35614.0,034757,34757,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,NaN,99 TAXIS,...,NaN,QUARTA-FEIRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
8,SP,35477.0,35614.0,030933,30933,MAQ VENDING BQ NW KREA,220,BEBIDAS QUENTES,5 ANDAR,99 TAXIS,...,NaN,QUARTA-FEIRA,X,X,X,X,X,NaN,NaN,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3459,SP,2922.0,75.0,8182,8182,MAQ VENDING BQ BIANCHI LEI 400,400,BEBIDAS QUENTES,BLOCO B 8° ANDAR,VIVO HUMBERTO,...,NaN,TERÇA-FEIRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
3460,SP,2922.0,75.0,8169,8169,MAQ VENDING BQ BIANCHI LEI 400,400,BEBIDAS QUENTES,BLOCO B 7° ANDAR,VIVO HUMBERTO,...,NaN,TERÇA-FEIRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
3462,SP,2922.0,75.0,8262,8262,MAQ VENDING BQ BIANCHI BVM 921,200,BEBIDAS QUENTES,RECEPCAO,VIVO OSASCO,...,NaN,TERÇA-FEIRA,NaN,X,NaN,X,NaN,NaN,NaN,2
3498,SP,41600.0,1019195.0,13518,13518,MAQ VENDING BQ BIANCHI LEI 400,400,BEBIDAS QUENTES,1 ANDAR BLOCO B,VOTORANTIM - LEOPOLDINA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5


In [54]:
rotas_sp_clean = rotas_sp_filtrado[["FILIAL", "PARCEIRO", "PATRIMÔNIO", "MODELO", "CLIENTE","CLIENTE AJUSTADO ÚNICO", "Latitude", "Longitude", "TIPO DE MAQUINA", "CAPACIDADE EM DOSES"]] #Colunas necessárias na planilha


### Aplicando padronização de strings

In [55]:
rotas_sp_clean['PARCEIRO'] = rotas_sp_clean['PARCEIRO'].apply(std_codes)
rotas_sp_clean['CLIENTE AJUSTADO ÚNICO'] = rotas_sp_clean['CLIENTE AJUSTADO ÚNICO'].apply(std_codes)

rotas_sp_clean=rotas_sp_clean.rename(columns={"PATRIMÔNIO": "PATRIMONIO"})
#rotas_sp_clean['PATRIMÔNIO'] = rotas_sp_clean['PATRIMÔNIO'].apply(std_patr)


C:\Users\tbekho01.ATKEARNEY_AD\AppData\Local\Temp\ipykernel_14340\290742962.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rotas_sp_clean['PARCEIRO'] = rotas_sp_clean['PARCEIRO'].apply(std_codes)
C:\Users\tbekho01.ATKEARNEY_AD\AppData\Local\Temp\ipykernel_14340\290742962.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rotas_sp_clean['CLIENTE AJUSTADO ÚNICO'] = rotas_sp_clean['CLIENTE AJUSTADO ÚNICO'].apply(std_codes)


In [56]:
rotas_sp_clean

,FILIAL,PARCEIRO,PATRIMONIO,MODELO,CLIENTE,CLIENTE AJUSTADO ÚNICO,Latitude,Longitude,TIPO DE MAQUINA,CAPACIDADE EM DOSES
4,SP,35614,028666,MAQ VENDING BQ NW KREA ONE TOUCH,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220
5,SP,35614,033465,MAQ VENDING BQ NW KREA ONE TOUCH,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220
6,SP,35614,28501,MAQ VENDING BQ NW KREA ONE TOUCH,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220
7,SP,35614,034757,MAQ VENDING BQ NW KREA ONE TOUCH,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220
8,SP,35614,030933,MAQ VENDING BQ NW KREA,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220
...,...,...,...,...,...,...,...,...,...,...
3459,SP,75,8182,MAQ VENDING BQ BIANCHI LEI 400,VIVO HUMBERTO,VIVO HUMBERTO 0,-23.587739,-46.640658,BEBIDAS QUENTES,400
3460,SP,75,8169,MAQ VENDING BQ BIANCHI LEI 400,VIVO HUMBERTO,VIVO HUMBERTO 0,-23.587739,-46.640658,BEBIDAS QUENTES,400
3462,SP,75,8262,MAQ VENDING BQ BIANCHI BVM 921,VIVO OSASCO,VIVO OSASCO 0,-23.530999,-46.785398,BEBIDAS QUENTES,200
3498,SP,1019195,13518,MAQ VENDING BQ BIANCHI LEI 400,VOTORANTIM - LEOPOLDINA,VOTORANTIM - LEOPOLDINA 0,-23.541408,-46.733697,BEBIDAS QUENTES,400


### Junção dos dados

In [57]:
capacidade_df_clean=capacidade_df.copy()
capacidade_df_clean = capacidade_df_clean[["MODELO", "NOMENCLATURA SIMPLIFICADA", "CAFÉ GRÃO", "LEITE", "CHOCOLATE", "CHÁ", "CAFÉ SOLÚVEL", "AÇÚCAR"]]

patrimonios_completo_capacidade = pd.merge(rotas_sp_clean, capacidade_df, on='MODELO', how='inner')
patrimonios_completo_capacidade=patrimonios_completo_capacidade.rename(columns={"CAPACIDADE EM DOSES": "COPOS"})

In [58]:
patrimonios_completo_capacidade

,FILIAL,PARCEIRO,PATRIMONIO,MODELO,CLIENTE,CLIENTE AJUSTADO ÚNICO,Latitude,Longitude,TIPO DE MAQUINA,COPOS,NOMENCLATURA SIMPLIFICADA,CAFÉ GRÃO,LEITE,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL,AÇÚCAR
0,SP,35614,028666,MAQ VENDING BQ NW KREA ONE TOUCH,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220,KREA TOUCH,1600,1000,1500,300,300,500
1,SP,35614,033465,MAQ VENDING BQ NW KREA ONE TOUCH,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220,KREA TOUCH,1600,1000,1500,300,300,500
2,SP,35614,28501,MAQ VENDING BQ NW KREA ONE TOUCH,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220,KREA TOUCH,1600,1000,1500,300,300,500
3,SP,35614,034757,MAQ VENDING BQ NW KREA ONE TOUCH,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220,KREA TOUCH,1600,1000,1500,300,300,500
4,SP,35614,030933,MAQ VENDING BQ NW KREA,99 TAXIS,99 TAXIS 0,-23.556438,-46.662896,BEBIDAS QUENTES,220,KREA,1600,1000,1500,300,300,500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544,SP,75,8182,MAQ VENDING BQ BIANCHI LEI 400,VIVO HUMBERTO,VIVO HUMBERTO 0,-23.587739,-46.640658,BEBIDAS QUENTES,400,LEI 400,2500,1000,3600,3300,1700,4000
545,SP,75,8169,MAQ VENDING BQ BIANCHI LEI 400,VIVO HUMBERTO,VIVO HUMBERTO 0,-23.587739,-46.640658,BEBIDAS QUENTES,400,LEI 400,2500,1000,3600,3300,1700,4000
546,SP,75,8262,MAQ VENDING BQ BIANCHI BVM 921,VIVO OSASCO,VIVO OSASCO 0,-23.530999,-46.785398,BEBIDAS QUENTES,200,BVM 921,1800,850,3000,1000,450,1700
547,SP,1019195,13518,MAQ VENDING BQ BIANCHI LEI 400,VOTORANTIM - LEOPOLDINA,VOTORANTIM - LEOPOLDINA 0,-23.541408,-46.733697,BEBIDAS QUENTES,400,LEI 400,2500,1000,3600,3300,1700,4000


### Preparando o arquivo

In [59]:
# Lista de colunas de insumos
colunas_insumos = ['COPOS', 'CAFÉ GRÃO', 'LEITE', 'CHOCOLATE', 'CHÁ', 'CAFÉ SOLÚVEL', 'AÇÚCAR']

# Usando melt para transformar as colunas de insumos em uma única coluna 'INSUMO'
insumos = pd.melt(patrimonios_completo_capacidade, 
                  id_vars=['FILIAL', 'CLIENTE AJUSTADO ÚNICO', 'PATRIMONIO', 'MODELO'], 
                  value_vars=colunas_insumos,
                  var_name='INSUMO', 
                  value_name='CAPACIDADE')

# Renomeando a coluna 'CLIENTE AJUSTADO ÚNICO' para 'PARCEIRO'
insumos['PARCEIRO'] = insumos['CLIENTE AJUSTADO ÚNICO']

# Adicionando a coluna 'NIVEL DE REPOSIÇÃO' com o valor fixo de 0,3
insumos['NIVEL_REPOSICAO'] = 0.3

# Removendo a coluna 'CLIENTE AJUSTADO ÚNICO', pois já temos 'PARCEIRO'
insumos = insumos.drop(columns=['CLIENTE AJUSTADO ÚNICO'])

# Convertendo a coluna 'CAPACIDADE' para numérico, forçando erros para NaN
insumos['CAPACIDADE'] = pd.to_numeric(insumos['CAPACIDADE'], errors='coerce')

# Removendo as linhas onde 'CAPACIDADE' não é numérica (NaN)
insumos = insumos.dropna(subset=['CAPACIDADE'])

# Reorganizando as colunas na ordem desejada
insumos = insumos[['FILIAL', 'PARCEIRO', 'PATRIMONIO', 'INSUMO', 'CAPACIDADE', 'NIVEL_REPOSICAO', 'MODELO']]

# Exibindo o resultado
insumos

,FILIAL,PARCEIRO,PATRIMONIO,INSUMO,CAPACIDADE,NIVEL_REPOSICAO,MODELO
0,SP,99 TAXIS 0,028666,COPOS,220.0,0.3,MAQ VENDING BQ NW KREA ONE TOUCH
1,SP,99 TAXIS 0,033465,COPOS,220.0,0.3,MAQ VENDING BQ NW KREA ONE TOUCH
2,SP,99 TAXIS 0,28501,COPOS,220.0,0.3,MAQ VENDING BQ NW KREA ONE TOUCH
3,SP,99 TAXIS 0,034757,COPOS,220.0,0.3,MAQ VENDING BQ NW KREA ONE TOUCH
4,SP,99 TAXIS 0,030933,COPOS,220.0,0.3,MAQ VENDING BQ NW KREA
...,...,...,...,...,...,...,...
3838,SP,VIVO HUMBERTO 0,8182,AÇÚCAR,4000.0,0.3,MAQ VENDING BQ BIANCHI LEI 400
3839,SP,VIVO HUMBERTO 0,8169,AÇÚCAR,4000.0,0.3,MAQ VENDING BQ BIANCHI LEI 400
3840,SP,VIVO OSASCO 0,8262,AÇÚCAR,1700.0,0.3,MAQ VENDING BQ BIANCHI BVM 921
3841,SP,VOTORANTIM - LEOPOLDINA 0,13518,AÇÚCAR,4000.0,0.3,MAQ VENDING BQ BIANCHI LEI 400


### Exportando a aba "insumos" de Dados.xlsx

In [60]:
# Caminho para o novo arquivo de saída
output_path = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Intermediários\Alan_Arthur\Dados.xlsx"

# Exportando o DataFrame 'insumos' para o novo arquivo Excel
insumos.to_excel(output_path, index=False, sheet_name="insumos")

# Mensagem de confirmação
print(f"Arquivo exportado com sucesso para: {output_path}")


Arquivo exportado com sucesso para: C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Intermediários\Alan_Arthur\Dados.xlsx


## 3) Consumo de insumos por patrimônio

Dados de Consumo.csv

### Importando dados

In [61]:
#Dado de gramaturas por doses
gramaturas_sku = pd.read_excel(arquivo_gramaturas, sheet_name='Gramaturas Kearney', skiprows=2, usecols="B:R")
patrimonios_sp = pd.read_excel(consumo_consolidado)

patrimonios_sp_clean = patrimonios_sp[["CODPARC", "NOMEPARC", "CODBEM", "CODPROD", "PRODUTO","CONSUMO DOSES", "Data"]]

patrimonios_sp_clean['NOMEPARC'] = patrimonios_sp_clean['NOMEPARC'].apply(std_codes)
#patrimonios_sp_clean['CODBEM'] = patrimonios_sp_clean['CODBEM'].apply(std_codes)
#patrimonios_sp_clean['CODBEM'] = patrimonios_sp_clean['CODBEM'].apply(std_patr)

In [62]:
patrimonios_sp_clean = patrimonios_sp_clean.rename(columns={"CONSUMO DOSES": "CONSUMO", "Data": "DATA"})

patrimonios_sp_clean['DATA'] = pd.to_datetime(patrimonios_sp_clean['DATA'])

# Create INICIO (first day of the month)
patrimonios_sp_clean['INICIO'] = patrimonios_sp_clean['DATA'].dt.to_period('M').dt.start_time

# Create FIM (last day of the month)
patrimonios_sp_clean['FIM'] = patrimonios_sp_clean['DATA'].dt.to_period('M').dt.end_time

# Drop the original DATA column
patrimonios_sp_clean = patrimonios_sp_clean.drop(columns=['DATA'])

# Format INICIO and FIM to show only DD/MM/YYYY
patrimonios_sp_clean['INICIO'] = patrimonios_sp_clean['INICIO'].dt.strftime('%d/%m/%Y')
patrimonios_sp_clean['FIM'] = patrimonios_sp_clean['FIM'].dt.strftime('%d/%m/%Y')

# Display result
patrimonios_sp_clean

,CODPARC,NOMEPARC,CODBEM,CODPROD,PRODUTO,CONSUMO,INICIO,FIM
0,6370.0,FUNDACAO BRADESCO,CMKSC000554,71,ÁGUA COM GÁS CRYSTAL PET 500ML,0.0,01/05/2025,31/05/2025
1,6370.0,FUNDACAO BRADESCO,3980,71,ÁGUA COM GÁS CRYSTAL PET 500ML,0.0,01/05/2025,31/05/2025
2,6370.0,FUNDACAO BRADESCO,4729,71,ÁGUA COM GÁS CRYSTAL PET 500ML,17.0,01/05/2025,31/05/2025
3,6370.0,FUNDACAO BRADESCO,4886,71,ÁGUA COM GÁS CRYSTAL PET 500ML,10.0,01/05/2025,31/05/2025
4,6370.0,FUNDACAO BRADESCO,4886,71,ÁGUA COM GÁS CRYSTAL PET 500ML,16.0,01/05/2025,31/05/2025
...,...,...,...,...,...,...,...,...
534223,NaN,HOSPITAL ALBERT EINSTEIN,MIX002417,599,DOSE DE MOCCACCINO EG,1.0,01/07/2025,31/07/2025
534224,NaN,HOSPITAL ALBERT EINSTEIN,MIX002417,581,DOSE DE ACHOCOLATADO G,7.0,01/07/2025,31/07/2025
534225,NaN,BANCO BRADESCO - DEP. JURIDICO,PR00362,511,DOSE DE CAFE EXPRESSO CURTO,2.0,01/07/2025,31/07/2025
534226,NaN,BANCO BRADESCO - DEP. JURIDICO,PR00362,512,DOSE DE CAFE EXPRESSO LONGO,1.0,01/07/2025,31/07/2025


In [63]:
merged_df = pd.merge(patrimonios_sp_clean, gramaturas_sku,
                         left_on='CODPROD', right_on='SKU', how='inner')

merged_df = merged_df.dropna(subset=['DOSE FINAL (mL)']).drop(["DOSE FINAL (mL)", "PALHETA", "Produto", "Obs", "Considerar como copo", "SKU"], axis =1)

insumos2 = ['CAFÉ GRÃO', 'LEITE', 'CHOCOLATE', 'CHÁ', 'CAFÉ SOLÚVEL', 'AÇÚCAR', "CAFÉ COM LEITE CARAMELO", "CAPPUCCINO COM CANELA CAFÉ DO CENTRO", "CAPPUCCINO SEM CANELA CAFÉ DO CENTRO", "COPO"]
for insumos_Item in insumos2:
    merged_df[insumos_Item] *= merged_df['CONSUMO']


In [64]:
# Somar as colunas especificadas para esses CODBEMs
soma_por_codbem = merged_df.groupby(['NOMEPARC','CODBEM', 'INICIO', 'FIM'])[['CAFÉ GRÃO', 'LEITE', 'CHOCOLATE', 'CHÁ', 'CAFÉ SOLÚVEL', 'AÇÚCAR', 'CAFÉ COM LEITE CARAMELO', 'CAPPUCCINO COM CANELA CAFÉ DO CENTRO', 'CAPPUCCINO SEM CANELA CAFÉ DO CENTRO', 'COPO']].sum().reset_index()

In [65]:
soma_por_codbem = soma_por_codbem.rename(columns={"COPO": "COPOS"})


In [66]:
soma_por_codbem

,NOMEPARC,CODBEM,INICIO,FIM,CAFÉ GRÃO,LEITE,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL,AÇÚCAR,CAFÉ COM LEITE CARAMELO,CAPPUCCINO COM CANELA CAFÉ DO CENTRO,CAPPUCCINO SEM CANELA CAFÉ DO CENTRO,COPOS
0,11 CARTORIO DE REG.,082001,01/05/2025,31/05/2025,10176.0,8053.0,1822.4,0.0,0.0,8480.0,0.0,0.0,0.0,1696.0
1,11 CARTORIO DE REG.,082001,01/06/2025,30/06/2025,7566.0,5389.9,2090.4,400.0,0.0,6305.0,0.0,0.0,0.0,1301.0
2,11 CARTORIO DE REG.,082001,01/07/2025,31/07/2025,5880.0,3197.3,5326.8,1760.0,0.0,4900.0,0.0,0.0,0.0,1220.0
3,11 CARTORIO DE REG.,4862,01/05/2025,31/05/2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,11 CARTORIO DE REG.,4862,01/06/2025,30/06/2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27176,[DASA] LABORATORIO VALECLIN,AMC00179,01/06/2025,30/06/2025,0.0,4108.5,1654.8,80.0,481.6,3660.0,0.0,0.0,0.0,741.0
27177,[DASA] LABORATORIO VALECLIN,AMC00179,01/07/2025,31/07/2025,0.0,3716.0,2875.6,60.0,462.6,3330.0,0.0,0.0,0.0,726.0
27178,[DASA] LABORATORIO VALECLIN,AMC00795,01/05/2025,31/05/2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27179,[DASA] LABORATORIO VALECLIN,AMC00795,01/06/2025,30/06/2025,0.0,0.0,0.0,4380.0,2054.8,9425.0,0.0,0.0,0.0,2323.0


In [67]:
melted_df = pd.melt(soma_por_codbem, 
                    id_vars=['NOMEPARC', 'CODBEM', 'INICIO', 'FIM'],  # Colunas que ficam fixas
                    value_vars=['CAFÉ GRÃO', 'LEITE', 'CHOCOLATE', 'CHÁ', 'CAFÉ SOLÚVEL', 
                                 'AÇÚCAR', 'CAFÉ COM LEITE CARAMELO', 
                                 'CAPPUCCINO COM CANELA CAFÉ DO CENTRO',
                                 'CAPPUCCINO SEM CANELA CAFÉ DO CENTRO', 'COPOS'],  # Colunas que vão virar INSUMO
                    var_name='INSUMO',  # Nome da nova coluna para as variáveis de ingredientes
                    value_name='CONSUMO')  # Nome da nova coluna para os valores

# Renomeando as colunas conforme a estrutura desejada
melted_df['FILIAL'] = "SP" # A coluna FILIAL será o CODPARC
melted_df['PARCEIRO'] = melted_df['NOMEPARC']  
melted_df['PATRIMONIO'] = melted_df['CODBEM']  

# Selecionando e reorganizando as colunas conforme a ordem desejada
final_df = melted_df[['FILIAL', 'PARCEIRO', 'PATRIMONIO', 'INSUMO', 'CONSUMO', 'INICIO', 'FIM']]

# Exibindo as primeiras linhas do DataFrame final
final_df = final_df.sort_values(by=['FILIAL', 'PARCEIRO', 'PATRIMONIO',"INICIO", "FIM", 'INSUMO'])

final_df

,FILIAL,PARCEIRO,PATRIMONIO,INSUMO,CONSUMO,INICIO,FIM
135905,SP,11 CARTORIO DE REG.,082001,AÇÚCAR,8480.0,01/05/2025,31/05/2025
163086,SP,11 CARTORIO DE REG.,082001,CAFÉ COM LEITE CARAMELO,0.0,01/05/2025,31/05/2025
0,SP,11 CARTORIO DE REG.,082001,CAFÉ GRÃO,10176.0,01/05/2025,31/05/2025
108724,SP,11 CARTORIO DE REG.,082001,CAFÉ SOLÚVEL,0.0,01/05/2025,31/05/2025
190267,SP,11 CARTORIO DE REG.,082001,CAPPUCCINO COM CANELA CAFÉ DO CENTRO,0.0,01/05/2025,31/05/2025
...,...,...,...,...,...,...,...
244628,SP,[DASA] LABORATORIO VALECLIN,AMC00795,CAPPUCCINO SEM CANELA CAFÉ DO CENTRO,0.0,01/07/2025,31/07/2025
81542,SP,[DASA] LABORATORIO VALECLIN,AMC00795,CHOCOLATE,0.0,01/07/2025,31/07/2025
108723,SP,[DASA] LABORATORIO VALECLIN,AMC00795,CHÁ,0.0,01/07/2025,31/07/2025
271809,SP,[DASA] LABORATORIO VALECLIN,AMC00795,COPOS,0.0,01/07/2025,31/07/2025


In [68]:
# ordenar por PATRIMONIO e INICIO
final_df = final_df.sort_values(["PATRIMONIO", "INICIO"])

# identificar o parceiro mais recente para cada PATRIMONIO
mais_recente = final_df.groupby("PATRIMONIO").tail(1)[["PATRIMONIO", "PARCEIRO"]]

# merge para marcar qual é o parceiro válido (o mais recente)
final_df = final_df.merge(mais_recente, on="PATRIMONIO", suffixes=("", "_RECENTE"))

# filtrar: só manter linhas em que PARCEIRO == PARCEIRO_RECENTE
final_df_filtrado = final_df[final_df["PARCEIRO"] == final_df["PARCEIRO_RECENTE"]].drop(columns=["PARCEIRO_RECENTE"])

In [69]:
final_df_filtrado

,FILIAL,PARCEIRO,PATRIMONIO,INSUMO,CONSUMO,INICIO,FIM
0,SP,KORDSA,001702,AÇÚCAR,2075.0,01/05/2025,31/05/2025
1,SP,KORDSA,001702,CAFÉ COM LEITE CARAMELO,0.0,01/05/2025,31/05/2025
2,SP,KORDSA,001702,CAFÉ GRÃO,2490.0,01/05/2025,31/05/2025
3,SP,KORDSA,001702,CAFÉ SOLÚVEL,0.0,01/05/2025,31/05/2025
4,SP,KORDSA,001702,CAPPUCCINO COM CANELA CAFÉ DO CENTRO,0.0,01/05/2025,31/05/2025
...,...,...,...,...,...,...,...
271805,SP,ITAU UNIBANCO,TTMA018439,CAPPUCCINO SEM CANELA CAFÉ DO CENTRO,0.0,01/07/2025,31/07/2025
271806,SP,ITAU UNIBANCO,TTMA018439,CHOCOLATE,499.7,01/07/2025,31/07/2025
271807,SP,ITAU UNIBANCO,TTMA018439,CHÁ,0.0,01/07/2025,31/07/2025
271808,SP,ITAU UNIBANCO,TTMA018439,COPOS,1451.0,01/07/2025,31/07/2025


In [70]:
# Primeiro, crie um dicionário de mapeamento de PATRIMONIO -> PARCEIRO
map_parceiro = dict(zip(insumos["PATRIMONIO"], insumos["PARCEIRO"]))

consumo = final_df_filtrado.copy()
# Atualize a coluna PARCEIRO em final_df_filtrado
consumo["PARCEIRO"] = consumo["PATRIMONIO"].map(map_parceiro)

consumo = consumo.dropna(subset=["PARCEIRO"])



In [71]:
consumo = consumo[['FILIAL', 'PARCEIRO', 'PATRIMONIO', 'INSUMO', 'CONSUMO', 'INICIO', 'FIM']]


In [72]:
consumo

,FILIAL,PARCEIRO,PATRIMONIO,INSUMO,CONSUMO,INICIO,FIM
600,SP,MITRE HEALTHY 0,016284,AÇÚCAR,1455.0,01/05/2025,31/05/2025
601,SP,MITRE HEALTHY 0,016284,CAFÉ COM LEITE CARAMELO,5494.5,01/05/2025,31/05/2025
602,SP,MITRE HEALTHY 0,016284,CAFÉ GRÃO,1746.0,01/05/2025,31/05/2025
603,SP,MITRE HEALTHY 0,016284,CAFÉ SOLÚVEL,0.0,01/05/2025,31/05/2025
604,SP,MITRE HEALTHY 0,016284,CAPPUCCINO COM CANELA CAFÉ DO CENTRO,0.0,01/05/2025,31/05/2025
...,...,...,...,...,...,...,...
271385,SP,TOKIO MARINE SAMPAIO 0,TTA019001,CAPPUCCINO SEM CANELA CAFÉ DO CENTRO,0.0,01/07/2025,31/07/2025
271386,SP,TOKIO MARINE SAMPAIO 0,TTA019001,CHOCOLATE,21328.7,01/07/2025,31/07/2025
271387,SP,TOKIO MARINE SAMPAIO 0,TTA019001,CHÁ,5040.0,01/07/2025,31/07/2025
271388,SP,TOKIO MARINE SAMPAIO 0,TTA019001,COPOS,3796.0,01/07/2025,31/07/2025


### Criando um ranking dos compartimentos de maior capacidade por patrimônio (para alocar os insumos especiais)

In [73]:

# 1. Filtrando os insumos desejados (CHÁ, CHOCOLATE, CAFÉ SOLÚVEL)
insumos_filtrados = insumos[insumos['INSUMO'].isin(['CHÁ', 'CHOCOLATE', 'CAFÉ SOLÚVEL'])]

# 2. Agrupando por PARCEIRO e PATRIMÔNIO, e ordenando por CAPACIDADE dentro de cada grupo
# Vamos usar sort_values para ordenar por CAPACIDADE
def ranking_insumos(group):
    # Classificando por CAPACIDADE em ordem decrescente (do maior para o menor)
    group_sorted = group.sort_values(by='CAPACIDADE', ascending=False)

    # Criando o ranking de 1 a 3 para os insumos dentro do grupo
    group_sorted['RANK'] = group_sorted['CAPACIDADE'].rank(ascending=False, method='first')

    # Pegando as colunas de RANK 1, 2, 3
    rank_1 = group_sorted[group_sorted['RANK'] == 1]['INSUMO'].values[0] if len(group_sorted[group_sorted['RANK'] == 1]) > 0 else None
    rank_2 = group_sorted[group_sorted['RANK'] == 2]['INSUMO'].values[0] if len(group_sorted[group_sorted['RANK'] == 2]) > 0 else None
    rank_3 = group_sorted[group_sorted['RANK'] == 3]['INSUMO'].values[0] if len(group_sorted[group_sorted['RANK'] == 3]) > 0 else None

    # Retornando a linha com o PARCEIRO, PATRIMÔNIO e os 3 insumos no ranking
    return pd.Series({
        'PARCEIRO': group['PARCEIRO'].values[0],
        'PATRIMONIO': group['PATRIMONIO'].values[0],
        'CAPACIDADE_1': rank_1,
        'CAPACIDADE_2': rank_2,
        'CAPACIDADE_3': rank_3
    })

# 3. Aplicando a função para cada grupo de PARCEIRO e PATRIMÔNIO
ranked_insumos = insumos_filtrados.groupby(['PARCEIRO', 'PATRIMONIO']).apply(ranking_insumos)

# 4. Resetando o índice do DataFrame final para uma estrutura mais limpa
ranked_insumos = ranked_insumos.reset_index(drop=True)

# Mostrando o DataFrame final com os rankings
ranked_insumos

C:\Users\tbekho01.ATKEARNEY_AD\AppData\Local\Temp\ipykernel_14340\136128671.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ranked_insumos = insumos_filtrados.groupby(['PARCEIRO', 'PATRIMONIO']).apply(ranking_insumos)


,PARCEIRO,PATRIMONIO,CAPACIDADE_1,CAPACIDADE_2,CAPACIDADE_3
0,99 TAXIS 0,028665,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL
1,99 TAXIS 0,028666,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL
2,99 TAXIS 0,029632,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL
3,99 TAXIS 0,030933,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL
4,99 TAXIS 0,033465,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL
...,...,...,...,...,...
523,VIVO HUMBERTO 0,8169,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL
524,VIVO HUMBERTO 0,8182,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL
525,VIVO OSASCO 0,8262,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL
526,VOTORANTIM - LEOPOLDINA 0,13518,CHOCOLATE,CHÁ,CAFÉ SOLÚVEL


### Alocando os insumos especiais nos compartimentos vazios dos patrimônios que possuem estes insumos

In [74]:

final_df_aux = consumo.copy()

# Supondo que final_df_aux e ranked_insumos já estão definidos
# final_df_aux é o dataframe principal, ranked_insumos é o dataframe com o ranking dos insumos

# Lista de insumos a serem verificados
target_insumos = ['CAFÉ COM LEITE CARAMELO', 
                  'CAPPUCCINO COM CANELA CAFÉ DO CENTRO',
                  'CAPPUCCINO SEM CANELA CAFÉ DO CENTRO']

# Passo 1: Filtrar as linhas onde o INSUMO é um dos alvos e o CONSUMO não é 0
mask_target_insumos = final_df_aux['INSUMO'].isin(target_insumos)
target_rows = final_df_aux[mask_target_insumos]

# Passo 2: Processar cada linha
rows_to_drop = []  # Para armazenar os índices das linhas a serem removidas

for idx, row in target_rows.iterrows():
    # Se CONSUMO for 0, ignorar e remover a linha
    if row['CONSUMO'] == 0:
        rows_to_drop.append(idx)
        continue

    # Passo 3: Verificar se CHÁ, CHOCOLATE ou CAFÉ SOLÚVEL têm CONSUMO igual a zero
    zero_consumption_ingredients = []
    
    for ingredient in ['CHÁ', 'CHOCOLATE', 'CAFÉ SOLÚVEL']:
        ingredient_row = final_df_aux[(final_df_aux['INSUMO'] == ingredient) & 
                                  #(final_df_aux['FILIAL'] == row['FILIAL']) & 
                                  (final_df_aux['PARCEIRO'] == row['PARCEIRO']) & 
                                  (final_df_aux['PATRIMONIO'] == row['PATRIMONIO']) & 
                                  (final_df_aux['INICIO'] == row['INICIO']) & 
                                  (final_df_aux['FIM'] == row['FIM'])]
        
        # Verificar se o CONSUMO é igual a zero e adicionar ao lista de insumos com consumo zero
        if not ingredient_row.empty and ingredient_row['CONSUMO'].values[0] == 0:
            zero_consumption_ingredients.append(ingredient)
    
    # Se mais de um insumo tiver CONSUMO igual a zero
    if len(zero_consumption_ingredients) > 0:
        # Passo 4: Verificar qual insumo tem o maior ranking de capacidade no ranked_insumos
        print(zero_consumption_ingredients)
        print(row['PATRIMONIO'])
        
        if row['PATRIMONIO'] in ranked_insumos["PATRIMONIO"].unique():
            print("s")
            insumo_cap_1 = ranked_insumos[ranked_insumos['PATRIMONIO'] == row['PATRIMONIO']]['CAPACIDADE_1'].values[0]
            insumo_cap_2 = ranked_insumos[ranked_insumos['PATRIMONIO'] == row['PATRIMONIO']]['CAPACIDADE_2'].values[0]
            insumo_cap_3 = ranked_insumos[ranked_insumos['PATRIMONIO'] == row['PATRIMONIO']]['CAPACIDADE_3'].values[0]
            
            if insumo_cap_1 in zero_consumption_ingredients:
                target_ingredient = insumo_cap_1 
            elif insumo_cap_2 in zero_consumption_ingredients:
                target_ingredient = insumo_cap_2
            elif insumo_cap_3 in zero_consumption_ingredients:
                target_ingredient = insumo_cap_3 

            ingredient_row = final_df_aux[(final_df_aux['INSUMO'] == target_ingredient) & 
                                #(final_df_aux['FILIAL'] == row['FILIAL']) & 
                                (final_df_aux['PARCEIRO'] == row['PARCEIRO']) & 
                                (final_df_aux['PATRIMONIO'] == row['PATRIMONIO']) & 
                                (final_df_aux['INICIO'] == row['INICIO']) & 
                                (final_df_aux['FIM'] == row['FIM'])]
            
            # Transferir o valor de CONSUMO para o insumo com maior ranking
            final_df_aux.loc[ingredient_row.index, 'CONSUMO'] = row['CONSUMO']
            final_df_aux.loc[ingredient_row.index, 'TRANSFER'] = row["INSUMO"]
            
            # Após a transferência, marcar a linha original para exclusão
            rows_to_drop.append(idx)
        else:
            print("n")
            rows_to_drop.append(idx)

# Passo 5: Deletar as linhas marcadas para exclusão
final_df_aux = final_df_aux.drop(rows_to_drop)

# Resetar o índice
final_df_aux = final_df_aux.reset_index(drop=True)

# Mostrar o DataFrame final
final_df_aux

['CHÁ', 'CAFÉ SOLÚVEL']
016284
s
['CHÁ', 'CAFÉ SOLÚVEL']
016284
s
['CHÁ', 'CAFÉ SOLÚVEL']
016284
s
['CAFÉ SOLÚVEL']
016292
s
['CAFÉ SOLÚVEL']
016292
s
['CAFÉ SOLÚVEL']
016292
s
['CAFÉ SOLÚVEL']
016295
s
['CAFÉ SOLÚVEL']
016295
s
['CAFÉ SOLÚVEL']
016295
s
['CHÁ', 'CAFÉ SOLÚVEL']
017481
s
['CHÁ', 'CAFÉ SOLÚVEL']
018592
s
['CAFÉ SOLÚVEL']
018592
s
['CHÁ', 'CAFÉ SOLÚVEL']
018592
s
['CAFÉ SOLÚVEL']
018592
s
['CHÁ', 'CAFÉ SOLÚVEL']
018592
s
['CAFÉ SOLÚVEL']
018592
s
['CAFÉ SOLÚVEL']
018597
s
['CAFÉ SOLÚVEL']
018597
s
['CHOCOLATE', 'CAFÉ SOLÚVEL']
018598
s
['CAFÉ SOLÚVEL']
018598
s
['CHÁ']
020143
s
['CHÁ']
020143
s
['CHÁ']
020143
s
['CAFÉ SOLÚVEL']
023552
s
['CAFÉ SOLÚVEL']
023552
s
['CAFÉ SOLÚVEL']
023552
s
['CAFÉ SOLÚVEL']
023553
s
['CAFÉ SOLÚVEL']
023553
s
['CAFÉ SOLÚVEL']
023553
s
['CAFÉ SOLÚVEL']
023556
s
['CAFÉ SOLÚVEL']
023556
s
['CAFÉ SOLÚVEL']
023556
s
['CAFÉ SOLÚVEL']
023558
s
['CAFÉ SOLÚVEL']
023558
s
['CAFÉ SOLÚVEL']
023558
s
['CAFÉ SOLÚVEL']
023564
s
['CAFÉ SOLÚVEL']
023564
s
['C

,FILIAL,PARCEIRO,PATRIMONIO,INSUMO,CONSUMO,INICIO,FIM,TRANSFER
0,SP,MITRE HEALTHY 0,016284,AÇÚCAR,1455.0,01/05/2025,31/05/2025,NaN
1,SP,MITRE HEALTHY 0,016284,CAFÉ GRÃO,1746.0,01/05/2025,31/05/2025,NaN
2,SP,MITRE HEALTHY 0,016284,CAFÉ SOLÚVEL,0.0,01/05/2025,31/05/2025,NaN
3,SP,MITRE HEALTHY 0,016284,CHOCOLATE,204.0,01/05/2025,31/05/2025,NaN
4,SP,MITRE HEALTHY 0,016284,CHÁ,5494.5,01/05/2025,31/05/2025,CAFÉ COM LEITE CARAMELO
...,...,...,...,...,...,...,...,...
10251,SP,TOKIO MARINE SAMPAIO 0,TTA019001,CAFÉ SOLÚVEL,11133.9,01/07/2025,31/07/2025,CAPPUCCINO COM CANELA CAFÉ DO CENTRO
10252,SP,TOKIO MARINE SAMPAIO 0,TTA019001,CHOCOLATE,21328.7,01/07/2025,31/07/2025,NaN
10253,SP,TOKIO MARINE SAMPAIO 0,TTA019001,CHÁ,5040.0,01/07/2025,31/07/2025,NaN
10254,SP,TOKIO MARINE SAMPAIO 0,TTA019001,COPOS,3796.0,01/07/2025,31/07/2025,NaN


In [75]:
#Retirando as linhas em que o consumo é 0
final_df_aux_export= final_df_aux[final_df_aux["CONSUMO"] != 0]

In [76]:
final_df_aux_export["PATRIMONIO"] = final_df_aux_export["PATRIMONIO"].astype(str)


C:\Users\tbekho01.ATKEARNEY_AD\AppData\Local\Temp\ipykernel_14340\1199123740.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df_aux_export["PATRIMONIO"] = final_df_aux_export["PATRIMONIO"].astype(str)


In [77]:
final_df_aux_export = final_df_aux_export.drop("TRANSFER", axis=1)


### Exportando Dados de Consumo.csv

In [78]:

# Caminho de saída
output_path = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Intermediários\Alan_Arthur\Dados de Consumo.csv"


# Exportar para CSV com encoding UTF-8
final_df_aux_export.to_csv(
    output_path,
    index=False,
    encoding='cp1252',
    sep=';',
    float_format="%.1f",  # garante duas casas decimais
    decimal='.'           # força . como decimal
)

# Mensagem de confirmação
print(f"Arquivo consolidado exportado com sucesso para: {output_path}")

Arquivo consolidado exportado com sucesso para: C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Intermediários\Alan_Arthur\Dados de Consumo.csv
